# L13 · Closed-Loop Evaluation and Capstone

This lab follows the six evidence gates from the lecture in one visible sequence: environment, dataset, checkpoint, frozen protocol, live rollouts, and a scoped report. Each step produces the evidence needed by the next one.

The core run evaluates one checkpoint for five Reference episodes. The optional Capstone then repeats the same seeds under one predeclared held-out condition. Five episodes are an integration smoke, not a stable performance benchmark.

## How to use this notebook

Work through the numbered steps from top to bottom.

1. Edit the configuration cell once. Set `checkpoint` to your own local path or pinned Hugging Face mapping; leave it as `None` to use the visible course default declared in that cell. Keep `dataset_root` pointed at the matching local dataset; if that directory is absent, Step 1 uses the visible pinned dataset fallback. Choose a fresh `run_name` whenever you repeat a formal run.
2. Run Steps 1–3 to audit the artifacts and freeze the comparison before seeing outcomes. Step 1 may download dataset metadata, but these steps do not download episodes or videos and do not start Genesis.
3. Continue when Step 1 prints `Ready to run Step 4: True`. Running Step 4 downloads or loads the checkpoint and immediately executes the Reference episodes.
4. Audit the videos and episode rows in Step 5.
5. To continue with the Capstone, run Step 6, then run Step 7 once to see its evidence card and sentence stems. Fill the three fields in that same cell and rerun only Step 7 to write the report.

Every experiment choice comes from the visible configuration. Running an evaluation cell is the explicit action that starts its subprocess, so Genesis never initializes in the notebook kernel.

In [ ]:
# Edit artifact paths and the declared experiment here.
CONFIG = {
    # A local pretrained_model directory, a pinned Hugging Face mapping, or None.
    # None uses COURSE_DEFAULT_CHECKPOINT below.
    # Example: {'repo_id': 'org/repo', 'revision': '<40-hex-commit>',
    #           'subfolder': 'checkpoints/020000/pretrained_model'}
    'checkpoint': None,
    # Prefer this local dataset when it exists. Otherwise Step 1 downloads
    # COURSE_DEFAULT_DATASET metadata from Hugging Face.
    'dataset_repo_id': 'genesis/fruit_pick',
    'dataset_root': 'datasets/fruit_pick',
    'run_name': 'fruit-pick-hf-fallback',
    'gpu_index': 0,
    'policy_device': 'cuda',
    'render': True,
    'protocol': {
        'episodes': 5,
        'base_seed': 1000,
        'control_fps': 100,
        'dwell_seconds': 0.4,
        'max_seconds': 15.0,
        'pick_objects': ('011_banana',),
        'place_target': '024_bowl',
        'success_tolerance_m': 0.06,
    },
    'held_out': {
        'name': 'shifted-friction-0_9',
        'factor': 'friction_ratio',
        'evaluation_range': (0.9, 0.9),
        'training_support': None,
    },
}

# Used only when CONFIG['dataset_root'] does not exist locally.
# L13 downloads meta/** only; policy evaluation does not read episodes or videos.
COURSE_DEFAULT_DATASET = {
    'repo_id': 'wangxun1010/fruit_pick',
    'revision': '475e8c6ec7079c4c52b20a0221bf959acb541abd',
    'subfolder': '.',
}

# Used only when CONFIG['checkpoint'] is None. The commit pin makes the download reproducible.
COURSE_DEFAULT_CHECKPOINT = {
    'repo_id': 'wangxun1010/smolvla-fruit-pick-0821',
    'revision': 'cdf720f162e6ee86dbc856cdf15c5dcd4a673221',
    'subfolder': '.',
}

## Step 1 — Audit the environment, dataset, and checkpoint source

The first three Capstone gates are identity checks. The dataset and checkpoint must agree on training dataset identity, camera features, state/action dimensions, FPS, processors, and any camera rename. Step 1 prefers `CONFIG['dataset_root']`; if that directory is absent, it downloads only `meta/**` from the pinned `COURSE_DEFAULT_DATASET`. A local checkpoint is audited now; a pinned checkpoint is reported as `DEFERRED` and audited after Step 4 downloads it.

This step reads metadata only. A pinned Hugging Face checkpoint is downloaded only when you run Step 4. The runtime summary also prints the exact child-process environment derived from `render`, `gpu_index`, and `policy_device`.

In [ ]:
import json

from robo_genesis.course_manifest import load_course_manifest
from robo_genesis.evaluation_contract import (
    evaluate_predicate_trace,
    validate_evaluation_protocol,
)
from robo_genesis.l13_workflow import (
    audit_l13_evidence,
    audit_l13_inputs,
    build_l13_capstone_guidance,
    capstone_statements,
    compare_l13_runs,
    declare_l13_conditions,
    prepare_l13_workflow,
    require_l13_inputs,
    run_l13_condition,
    runtime_summary,
    write_l13_capstone_report,
)

lesson = load_course_manifest().lesson('L13')
assert lesson.status.value == 'gpu-verified'
workflow = prepare_l13_workflow(
    CONFIG,
    course_default_dataset=COURSE_DEFAULT_DATASET,
    course_default_checkpoint=COURSE_DEFAULT_CHECKPOINT,
)
preflight = audit_l13_inputs(
    workflow, allow_dataset_download=True, allow_checkpoint_download=False
)

print(json.dumps(runtime_summary(workflow, lesson_status=lesson.status.value), indent=2, default=str))
print(json.dumps(list(preflight.rows), indent=2))
print('Ready to run Step 4:', preflight.ready_for_reference)
if not preflight.ready_for_reference:
    print("STOP before Step 4 — edit CONFIG above and fix every FAIL row.")

## Step 2 — Freeze the task and success protocol

The protocol is fixed before any rollout result is visible. Success requires the L08 containment predicate to remain true for `ceil(dwell_seconds × policy_fps)` consecutive checks; a final true frame at timeout does not bypass this dwell rule.

**Protocol exercise:** edit `SYNTHETIC_PREDICATE_TRACE`, predict its maximum streak and outcome on paper, then run the cell. Include a trace whose last frame is true but whose streak is too short.

In [ ]:
AVAILABLE_OBJECTS = (
    '003_cracker_box', '004_sugar_box', '005_tomato_soup_can',
    '006_mustard_bottle', '007_tuna_fish_can', '008_pudding_box',
    '009_gelatin_box', '010_potted_meat_can', '011_banana',
    '019_pitcher_base', '021_bleach_cleanser', '024_bowl',
)
protocol_config = CONFIG['protocol']
policy_fps = float(preflight.dataset['fps']) if preflight.dataset else 5.0
reference_protocol = validate_evaluation_protocol(
    episodes=protocol_config['episodes'],
    policy_fps=policy_fps,
    control_fps=protocol_config['control_fps'],
    dwell_seconds=protocol_config['dwell_seconds'],
    max_seconds=protocol_config['max_seconds'],
    tolerance=protocol_config['success_tolerance_m'],
    pick_objects=protocol_config['pick_objects'],
    place_target=protocol_config['place_target'],
    available_objects=AVAILABLE_OBJECTS,
    base_seed=protocol_config['base_seed'],
    balanced=True,
    condition_name='reference',
    render_enabled=workflow.render_enabled,
)

SYNTHETIC_PREDICATE_TRACE = [False, True, False, True]
trace_evidence = evaluate_predicate_trace(
    SYNTHETIC_PREDICATE_TRACE,
    dwell_seconds=protocol_config['dwell_seconds'],
    policy_fps=policy_fps,
)
print(json.dumps({
    'task': reference_protocol['task'],
    'timing': reference_protocol['timing'],
    'success_rule': reference_protocol['success_rule'],
    'episode_seeds': reference_protocol['sampling']['episode_seeds'],
    'object_schedule': reference_protocol['sampling']['object_schedule'],
    'synthetic_trace': trace_evidence,
}, indent=2))

## Step 3 — Predeclare the Reference and one held-out condition

The Capstone comparison changes exactly one factor while holding the task, camera contract, object schedule, and episode seeds fixed. The label is relative to training provenance: missing or overlapping support remains `shifted`; only documented disjoint support may be called OOD.

**One-factor exercise:** change only `CONFIG['held_out']`. Before running anything, explain the resulting label and name every factor that remains fixed.

In [ ]:
reference_condition, held_out_condition = declare_l13_conditions(CONFIG['held_out'])
condition_plan = {
    'reference': reference_condition,
    'held_out': held_out_condition or 'SKIP — no changed factor was declared',
}
print(json.dumps(condition_plan, indent=2))

## Step 4 — Run the five-episode Reference evaluation

This is the first expensive step. Run this cell only after Steps 1–3 look correct. It downloads the pinned checkpoint when needed, launches `robo_genesis.eval_policy` in a separate process, captures the full log, maps the selected GPU, and requires fresh result and video targets. An evaluator crash stops the cell instead of becoming a policy failure.

In [ ]:
reference_inputs = require_l13_inputs(workflow)
reference_run = run_l13_condition(
    workflow, reference_inputs, reference_protocol, reference_condition
)
print(json.dumps({
    'status': 'COMPLETED',
    'results': str(reference_run.paths['results']),
    'videos': str(reference_run.paths['videos']),
    'summary': reference_run.report['summary'],
}, indent=2))

## Step 5 — Audit the episode rows and rollout videos

The stored summary is accepted only if `k/n`, Wilson interval, termination counts, per-object counts, seeds, protocol, and condition can be recomputed from the episode rows. Every episode must also have a decodable H.264 video with valid dimensions and frame count.

**Failure-evidence exercise:** after a real run, label one failed seed using a selection rule chosen before watching it. Record a short category and evidence visible in that rollout; do not infer a training-data or architecture cause from the aggregate rate.

In [ ]:
MANUAL_FAILURE_LABELS = {
    # 1003: {
    #     'category': 'release timing',
    #     'visible_evidence': 'the object leaves the gripper beside the bowl',
    # },
}
reference_evidence = audit_l13_evidence(
    reference_run.report, manual_failure_labels=MANUAL_FAILURE_LABELS
)
print(json.dumps({
    'summary': reference_evidence['summary'],
    'videos': reference_evidence['videos'],
    'selection_rule': reference_evidence['selection_rule'],
    'manual_failure_labels': reference_evidence['manual_failure_labels'],
}, indent=2))
from IPython.display import Video, display
display(Video(reference_evidence['representative_episode']['video'], embed=True))

## Step 6 — Run the matched held-out comparison

Run this cell only after the Reference evidence is complete and when you want to continue the Capstone. It reuses the checkpoint, task, object schedule, and five seeds while changing the one factor declared in Step 3. Both aggregate `k/n` values and every seed-level outcome flip are reported. Five matched pairs guide diagnosis; they do not establish statistical significance.

In [ ]:
if held_out_condition is None:
    raise RuntimeError('Declare one held-out factor in Step 3 before Step 6')
held_out_run = run_l13_condition(
    workflow, reference_inputs, reference_protocol, held_out_condition
)
held_out_evidence = audit_l13_evidence(held_out_run.report)
paired_changes = compare_l13_runs(reference_run, held_out_run)
print(json.dumps({
    'reference': reference_evidence['summary'],
    held_out_condition['name']: held_out_evidence['summary'],
    'paired_outcome_changes': paired_changes,
    'interpretation': 'Matched outcomes; no significance claim is made.',
}, indent=2))

## Step 7 — Write the Capstone conclusion from the evidence chain

The report answers one scoped question and links it back to the runtime, dataset, checkpoint, frozen protocol, episode rows, videos, limitations, and reproduction commands. Run this cell once after Step 6: it prints an evidence card and sentence stems derived from the actual results. Then fill the three empty fields in `CAPSTONE_CONCLUSION` inside this same cell and rerun only Step 7; the rollouts are not repeated.

**Capstone exercise:** use the evidence card to state what the run supports, what it cannot support, and which one next experiment would reduce the most important remaining uncertainty. Do not invent a causal explanation that is absent from the episode rows and videos.

In [ ]:
reference_summary = reference_evidence['summary']
held_out_summary = held_out_evidence['summary']
guidance = build_l13_capstone_guidance(
    reference_summary, held_out_summary, held_out_condition, paired_changes
)
print('EVIDENCE CARD — base every conclusion on these observed values')
print(json.dumps(guidance['evidence_card'], indent=2))
print('\nSENTENCE STEMS — edit these into your own scoped conclusions')
print(json.dumps(guidance['sentence_stems'], indent=2))

# First run: leave these blank and read the evidence card above.
# Then replace all three blanks with your conclusions and rerun only this cell.
CAPSTONE_CONCLUSION = {
    'question': guidance['question'],
    'supported_claim': '',
    'unsupported_claim': '',
    'next_evidence': '',
}
required_fields = ('supported_claim', 'unsupported_claim', 'next_evidence')
missing_fields = [
    name for name in required_fields if not CAPSTONE_CONCLUSION[name].strip()
]
statements = capstone_statements(CAPSTONE_CONCLUSION)
capstone_output = None
if missing_fields:
    print("\nACTION REQUIRED — Step 7 is a two-pass cell.")
    print('Fill these CAPSTONE_CONCLUSION fields:', ', '.join(missing_fields))
    print('Use the exact k/n and matched changes above; do not invent causes.')
    print('After editing this cell, rerun only Step 7. Rollouts are not repeated.')
else:
    capstone_output = write_l13_capstone_report(
        workflow, reference_run, held_out_run, paired_changes, statements
    )
    print(capstone_output['markdown'])

core_complete = bool(
    reference_summary['episodes'] == protocol_config['episodes']
    and reference_run.report['protocol']['render_enabled']
    and reference_run.report['environment'].get('actual_policy_device', '').startswith('cuda')
    and len(reference_evidence['videos']) == protocol_config['episodes']
)
capstone_complete = bool(
    core_complete
    and len(held_out_evidence['videos']) == protocol_config['episodes']
    and len(paired_changes) == protocol_config['episodes']
    and capstone_output is not None
)
print('Step 1 preflight: INPUTS READY')
print('Core Reference evaluation:', 'COMPLETED' if core_complete else 'NOT COMPLETED')
print('Capstone comparison and report:', 'COMPLETED' if capstone_complete else 'NOT COMPLETED')

## Beyond the core lab — checkpoint sweep

The lecture describes `python -m robo_genesis.eval_sweep` as an optional terminal extension. It is intentionally not another notebook branch: its cost is `number of checkpoints × episodes`, it is not required for the single-checkpoint Capstone, and one checkpoint is one point rather than a learning curve.